# Supervised Learning: Adult Income Classification

## Objective
Build a supervised learning classification model on the UCI Adult (Census Income) dataset to predict whether an individual's annual income is `<=50K` or `>50K`.

### Workflow
1. Load the public UCI Adult dataset.
2. Inspect and clean the data.
3. Engineer numerical features using log transformation for highly skewed capital-gain and capital-loss.
4. Encode categorical variables and standardize numerical variables.
5. Split the data into training and testing sets using stratification.
6. Use 5-fold cross-validation and GridSearchCV to tune Logistic Regression.
7. Evaluate the final model using accuracy, precision, recall, F1-score, ROC-AUC and a confusion matrix.
8. Analyze important model coefficients and discuss strengths and limitations.

**Important:** The target variable is used only as the prediction target; it is not included in the feature matrix.


In [ ]:
# Install the UCI package automatically if needed
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("ucimlrepo") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ucimlrepo", "-q"])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
print("Libraries imported successfully.")


In [ ]:
# Load the UCI Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

# Clean column names and target labels
X.columns = X.columns.astype(str).str.strip()
y = y.iloc[:, 0].astype(str).str.strip().str.replace(".", "", regex=False)

print("Dataset shape:", X.shape)
print("\nFeature columns:")
print(list(X.columns))
print("\nTarget distribution:")
print(y.value_counts())


In [ ]:
# Basic inspection
print("Data types:")
print(X.dtypes)

print("\nMissing-value markers before treatment:")
marker_counts = {}
for col in X.columns:
    marker_counts[col] = X[col].astype(str).str.strip().eq("?").sum()
print(pd.Series(marker_counts).sort_values(ascending=False))

print("\nFirst five rows:")
display(X.head())


## Data Cleaning and Feature Engineering

The Adult dataset contains missing-value markers represented by `?`. These are converted to missing values. Numerical variables are imputed with their median, while categorical variables are imputed with their most frequent value.

`capital-gain` and `capital-loss` are strongly right-skewed, so `log1p` transformation is applied before standardization. Categorical variables are one-hot encoded. All preprocessing is placed inside a Scikit-learn pipeline so that transformations are learned only from the training folds during cross-validation, reducing data leakage risk.


In [ ]:
# Convert '?' markers to NaN
X = X.replace(r"^\s*\?$", np.nan, regex=True)

numeric_features = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

categorical_features = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country"
]

# Convert numeric columns explicitly
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Feature engineering: log-transform skewed monetary variables
X["capital-gain"] = np.log1p(X["capital-gain"])
X["capital-loss"] = np.log1p(X["capital-loss"])

print("Missing values after marker conversion:", int(X.isna().sum().sum()))
print("Feature matrix shape:", X.shape)


In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set :", X_test.shape, y_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))


In [ ]:
# Version-compatible OneHotEncoder
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = LogisticRegression(
    max_iter=2000,
    solver="liblinear",
    random_state=RANDOM_STATE
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

print("Pipeline created successfully.")


## Model Selection with 5-Fold Cross-Validation

Logistic Regression was selected because this is a binary classification problem, the model provides a strong interpretable baseline, and its regularization parameter can be tuned systematically. Five-fold stratified cross-validation is used on the training data, while the test set remains untouched until final evaluation.


In [ ]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best C:", grid.best_params_["model__C"])
print("Best mean CV ROC-AUC:", round(grid.best_score_, 4))

cv_results = pd.DataFrame(grid.cv_results_)[[
    "param_model__C", "mean_train_score", "mean_test_score",
    "std_test_score", "rank_test_score"
]]
display(cv_results.sort_values("rank_test_score"))


### Validation scoring note
The positive class is `>50K`. Explicit `pos_label=">50K"` is used for precision, recall and F1 so that these metrics are computed correctly for string target labels.


In [ ]:
# Detailed 5-fold cross-validation using the selected model
best_model = grid.best_estimator_

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=">50K"),
    "recall": make_scorer(recall_score, pos_label=">50K"),
    "f1": make_scorer(f1_score, pos_label=">50K"),
    "roc_auc": "roc_auc"
}

cv_scores = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False
)

cv_summary = pd.DataFrame({
    metric: [
        cv_scores[f"test_{metric}"].mean(),
        cv_scores[f"test_{metric}"].std()
    ]
    for metric in scoring
}, index=["mean", "std"]).T

display(cv_summary.round(4))


In [ ]:
# Fit the selected model on all training data
best_model.fit(X_train, y_train)

# Final test predictions
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, pos_label=">50K"),
    "Recall": recall_score(y_test, y_pred, pos_label=">50K"),
    "F1-score": f1_score(y_test, y_pred, pos_label=">50K"),
    "ROC-AUC": roc_auc_score((y_test == ">50K").astype(int), y_prob)
}

print("Final Test Metrics")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=["<=50K", ">50K"])
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["<=50K", ">50K"]
)
disp.plot()
plt.title("Logistic Regression - Test Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob, pos_label=">50K")
plt.title("Logistic Regression - ROC Curve")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect model coefficients
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = best_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values("abs_coefficient", ascending=False)

print("Top 20 features by absolute coefficient:")
display(coef_df.head(20))

plt.figure(figsize=(10, 7))
top = coef_df.head(15).sort_values("coefficient")
plt.barh(top["feature"], top["coefficient"])
plt.xlabel("Logistic Regression Coefficient")
plt.title("Top 15 Most Influential Features")
plt.tight_layout()
plt.show()


## Interpretation

- **Accuracy** measures the proportion of all test observations classified correctly.
- **Precision** measures how many observations predicted as `>50K` were actually `>50K`.
- **Recall** measures how many actual `>50K` observations were successfully detected.
- **F1-score** balances precision and recall.
- **ROC-AUC** measures the model's ability to rank positive cases above negative cases across classification thresholds.
- The confusion matrix shows the types of correct and incorrect predictions.
- Logistic Regression coefficients provide an interpretable indication of which transformed/encoded features are associated with the predicted class. Coefficients should be interpreted as model associations, not causal effects.

Because the target classes are imbalanced, accuracy alone should not be used to judge the model. Precision, recall, F1-score and ROC-AUC provide additional information about minority-class performance.


In [ ]:
# Compact final summary for the report
print("===== FINAL SUMMARY =====")
print("Dataset:", X.shape[0], "records")
print("Processed feature count after one-hot encoding:",
      len(best_model.named_steps["preprocessor"].get_feature_names_out()))
print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Best C:", grid.best_params_["model__C"])
print("Mean 5-fold CV ROC-AUC:", round(grid.best_score_, 4))
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")
print("=========================")
